# 🏦 Banking Intent Classification — Video Demo

**Mô hình:** Llama-3.2-3B-Instruct + QLoRA (Unsloth)

**Dataset:** BANKING77 — 77 intent categories

Video demo theo yêu cầu bài lab:
1. Cách chạy inference script
2. Input message mẫu
3. Predicted intent label
4. Final accuracy trên test set

In [ ]:
# =============================================
# CELL 1: Config
# =============================================
GITHUB_USERNAME = "tzin1401"
REPO_NAME = "banking-intent-unsloth"
BRANCH = "main"
OUTPUT_DATASET_NAME = "banking-intent-trained-model"

REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
print("📋 Config:")
print(f"   Repo: {REPO_URL}")
print(f"   Dataset: {OUTPUT_DATASET_NAME}")

In [ ]:
# =============================================
# CELL 2: Install dependencies
# =============================================
!pip install --no-deps xformers trl peft accelerate bitsandbytes triton -q
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install "transformers<=5.5.0" "datasets<4.4.0" "dill>=0.3.9" pyyaml scikit-learn -q
print("\n✅ Dependencies installed")

In [ ]:
# =============================================
# CELL 3a: DEBUG — Xem cấu trúc /kaggle/input/
# =============================================
# Chạy cell này TRƯỚC để biết chính xác dataset nằm ở đâu
import os, glob

print("📂 Cấu trúc /kaggle/input/ (3 tầng đầu):")
print("=" * 50)
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)
    if depth < 3:
        indent = "  " * depth
        print(f"{indent}📁 {os.path.basename(root)}/")
        # Show files at this level
        for fn in sorted(files)[:10]:
            print(f"{indent}  📄 {fn}")
        if len(files) > 10:
            print(f"{indent}  ... và {len(files)-10} files khác")

print("\n" + "=" * 50)
print("\n🔍 Tìm adapter_config.json:")
found = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
for f in found:
    print(f"   ✅ {f}")
if not found:
    print("   ❌ Không tìm thấy!")

print("\n🔍 Tìm eval_results.txt:")
found2 = glob.glob("/kaggle/input/**/eval_results.txt", recursive=True)
for f in found2:
    print(f"   ✅ {f}")

print("\n🔍 Tìm labels.txt:")
found3 = glob.glob("/kaggle/input/**/labels.txt", recursive=True)
for f in found3:
    print(f"   ✅ {f}")

In [ ]:
# =============================================
# CELL 3b: Clone repo + Auto-detect & Copy model
# =============================================
import os, shutil, glob

# Clone repo
os.environ["REPO_URL"] = REPO_URL
os.environ["REPO_NAME"] = REPO_NAME
os.environ["BRANCH"] = BRANCH

if not os.path.exists(REPO_NAME):
    !git clone "$REPO_URL"
%cd "$REPO_NAME"
!git pull origin "$BRANCH" -q

# --- Auto-detect: tìm adapter_config.json trong toàn bộ /kaggle/input/ ---
adapter_files = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)

if not adapter_files:
    print("❌ Không tìm thấy adapter_config.json trong /kaggle/input/")
    print("   Hãy chạy Cell 3a ở trên để xem cấu trúc thư mục.")
    raise FileNotFoundError("adapter_config.json not found")

# Lấy thư mục chứa adapter_config.json (đó chính là outputs dir)
# Ưu tiên file KHÔNG nằm trong checkpoint-* (đó là model cuối cùng)
src_outputs = None
for af in adapter_files:
    parent = os.path.dirname(af)
    if "checkpoint-" not in parent:
        src_outputs = parent
        break
if src_outputs is None:
    # Fallback: dùng file đầu tiên
    src_outputs = os.path.dirname(adapter_files[0])

print(f"✅ Tìm thấy model tại: {src_outputs}")

# --- Auto-detect: tìm labels.txt ---
labels_files = glob.glob("/kaggle/input/**/labels.txt", recursive=True)
src_data = None
if labels_files:
    src_data = os.path.dirname(labels_files[0])
    print(f"✅ Tìm thấy data tại: {src_data}")

# --- Copy outputs ---
if os.path.exists("./outputs"):
    shutil.rmtree("./outputs")
shutil.copytree(src_outputs, "./outputs")
print(f"   📁 Copied → ./outputs/ ({len(os.listdir('./outputs'))} items)")

# --- Copy sample_data ---
if src_data:
    if os.path.exists("./sample_data"):
        shutil.rmtree("./sample_data")
    shutil.copytree(src_data, "./sample_data")
    print(f"   📁 Copied → ./sample_data/ ({len(os.listdir('./sample_data'))} items)")

# --- Verify ---
print("\n📦 Kiểm tra:")
checks = {
    "adapter_config.json": os.path.exists("./outputs/adapter_config.json"),
    "adapter_model.*": bool(glob.glob("./outputs/adapter_model*")),
    "tokenizer.json": os.path.exists("./outputs/tokenizer.json"),
    "eval_results.txt": os.path.exists("./outputs/eval_results.txt"),
    "labels.txt": os.path.exists("./sample_data/labels.txt") if src_data else False,
    "test.csv": os.path.exists("./sample_data/test.csv") if src_data else False,
}
for name, ok in checks.items():
    print(f"   {'✅' if ok else '❌'} {name}")

print("\n✅ Sẵn sàng!")

---
## 📄 Phần 1: Giới thiệu Inference Script

File `scripts/inference.py` implement class `IntentClassification` theo đúng interface yêu cầu đề bài.

In [ ]:
# =============================================
# CELL 4: Xem code inference.py
# =============================================
print("=" * 60)
print("  📄 FILE: scripts/inference.py")
print("=" * 60)
print()
with open("scripts/inference.py", "r") as f:
    content = f.read()
print(content)
print("=" * 60)

In [ ]:
# =============================================
# CELL 5: Xem config inference.yaml
# =============================================
print("=" * 60)
print("  ⚙️  FILE: configs/inference.yaml")
print("=" * 60)
print()
with open("configs/inference.yaml", "r") as f:
    print(f.read())
print("─" * 60)
print("Giải thích:")
print("  • model_path:     đường dẫn đến LoRA adapter đã train")
print("  • base_model:     Llama-3.2-3B-Instruct quantized 4-bit")
print("  • max_seq_length: 768 tokens (khớp với training config)")
print("  • labels_path:    file chứa 77 intent labels hợp lệ")
print("  • temperature:    0.0 → deterministic (greedy decoding)")
print("─" * 60)

---
## 🔧 Phần 2: Load Model + Chạy Inference

In [ ]:
# =============================================
# CELL 6: Load model bằng IntentClassification class
# =============================================
# Cách sử dụng (2 bước):
#   Bước 1: classifier = IntentClassification("configs/inference.yaml")
#   Bước 2: label = classifier("customer message here")

import sys
sys.path.insert(0, ".")

from scripts.inference import IntentClassification

print("🔄 Đang load model từ ./outputs/ ...")
print("   Config: configs/inference.yaml")
print("   Base model: Llama-3.2-3B-Instruct-bnb-4bit")
print()
classifier = IntentClassification("configs/inference.yaml")
print("\n✅ Model đã load xong!")
print()
print("Cách dùng:")
print('  label = classifier("I am still waiting on my card?")')
print('  print(label)  # → card_arrival')

---
## 🏷️ Phần 3: Demo Inference — Dự đoán Intent

In [ ]:
# =============================================
# CELL 7: Demo inference với các message mẫu
# =============================================

demo_messages = [
    "I am still waiting on my card?",
    "How do I change my PIN number?",
    "I want to cancel my transfer",
    "Why was I charged extra for my card payment?",
    "How do I top up my account using a bank transfer?",
    "I lost my card, what should I do?",
    "Is my card compatible with Apple Pay?",
    "My transfer hasn't arrived yet",
]

print("=" * 60)
print("  🏦 Banking Intent Classification — Inference Demo")
print("=" * 60)
print()

for i, msg in enumerate(demo_messages, 1):
    label = classifier(msg)
    print(f"  [{i}] 📩 Input:  \"{msg}\"")
    print(f"      🏷️  Output: {label}")
    print()

print("=" * 60)
print(f"  ✅ Đã dự đoán thành công {len(demo_messages)} messages!")
print("=" * 60)

---
## 🧪 Phần 4: Thử nghiệm tự do — Custom Input

In [ ]:
# =============================================
# CELL 8: Thử với message tự nhập
# =============================================

custom_message = "I need to make a payment but my card is blocked"

print(f"📩 Input:  \"{custom_message}\"")
result = classifier(custom_message)
print(f"🏷️  Predicted Intent: {result}")

In [ ]:
# =============================================
# CELL 9: Thêm một vài ví dụ nữa
# =============================================

more_examples = [
    "Can you help me get a refund?",
    "I want to open a savings account",
    "What are the exchange rates today?",
]

print("🔍 Thử thêm một số message khác:\n")
for msg in more_examples:
    label = classifier(msg)
    print(f"  📩 \"{msg}\"")
    print(f"  🏷️  → {label}\n")

---
## 📊 Phần 5: Kết quả Accuracy trên Test Set

In [ ]:
# =============================================
# CELL 10: Hiển thị kết quả accuracy
# =============================================
import os

eval_path = "./outputs/eval_results.txt"

print("=" * 60)
print("  📊 KẾT QUẢ ĐÁNH GIÁ TRÊN TEST SET")
print("=" * 60)

if os.path.exists(eval_path):
    with open(eval_path, "r") as f:
        content = f.read()
    print(content)
else:
    print("⚠️  File eval_results.txt chưa có.")
    print("   File này được tạo khi chạy train.py (đánh giá trên test set).")

print("=" * 60)

---
## 📋 Tổng kết

- **Model**: Llama-3.2-3B-Instruct fine-tuned với QLoRA 4-bit (Unsloth)
- **Dataset**: BANKING77 — 77 intent categories, ~10K training samples
- **Kỹ thuật**: LoRA r=64, alpha=128, cosine scheduler, 6 epochs
- **Inference**: Alpaca-style prompt + fuzzy label matching

Kết quả chi tiết xem tại `outputs/eval_results.txt`.